In [ ]:
%%file app.py
from flask import Flask

app = Flask(__name__)

@app.route("/")                      # URL: http://localhost:5000/
def home():
    return "Welcome to the transaction monitoring system!"

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

In [ ]:
import subprocess, time, requests

server = subprocess.Popen(["python", "app.py"])
time.sleep(2)   # give the server a moment to start

response = requests.get("http://localhost:5000/")
print(f"Status: {response.status_code}")
print(f"Body:   {response.text}")

In [ ]:
server.kill()

In [ ]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/")
def home():
    return "Welcome to the transaction monitoring system!"

@app.route("/hello")                  # GET /hello?name=Anna
def hello():
    name = request.args.get("name", "stranger")   # read query parameter
    return f"Hello, {name}!"

@app.route("/transaction/<tx_id>")    # GET /transaction/TX0042
def get_transaction(tx_id):
    return jsonify({"tx_id": tx_id, "status": "found"})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

In [ ]:
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

# Query parameter
r1 = requests.get("http://localhost:5000/hello")
r2 = requests.get("http://localhost:5000/hello?name=Anna")
print(r1.text)   # Hello, stranger!
print(r2.text)   # Hello, Anna!

# Path variable
r3 = requests.get("http://localhost:5000/transaction/TX0042")
print(r3.json())

In [ ]:
server.kill()

In [ ]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/")
def home():
    return "Welcome to the transaction monitoring system!"

@app.route("/hello")
def hello():
    name = request.args.get("name", "stranger")
    return f"Hello, {name}!"

@app.route("/transaction/<tx_id>")
def get_transaction(tx_id):
    return jsonify({"tx_id": tx_id, "status": "found"})

@app.route("/status")
def status():
    return jsonify({
        "service": "monitoring",
        "version": "1.0",
        "status": "ok"
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

In [ ]:
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

r = requests.get("http://localhost:5000/status")
print(r.json())

In [ ]:
server.kill()

In [ ]:
!pkill -f app.py

In [ ]:
import subprocess, time

server = subprocess.Popen(["python", "app.py"])

time.sleep(3)

In [ ]:
r = requests.post(
    "http://localhost:5000/echo",
    json={
        "tx_id": "TX0042",
        "amount": 4500.0,
        "store": "Krakow",
        "category": "electronics"
    }
)

print(r.status_code)
print(r.json())

In [ ]:
###3.1
r_bad = requests.get("http://localhost:5000/echo")
print(f"Status: {r_bad.status_code}")

In [ ]:
# Answer in a comment: what does this status code mean?
# ANSWER: # Status code 405 means "Method Not Allowed".
# The endpoint /echo only accepts POST requests.
# A GET request is rejected by Flask because that HTTP method is not permitted for this route.

In [ ]:
#Part 4: Transaction Scoring via API
server.kill()

In [ ]:


def score_transaction(tx: dict) -> dict:
    """
    Assess transaction risk using business rules.
    Returns dict with: score (0-8), risk_level, triggered_rules.
    """
    score = 0
    rules = []

    if tx.get("amount", 0) > 3000:
        score += 3; rules.append("R1: amount > 3000")

    if tx.get("category") == "electronics" and tx.get("amount", 0) > 1500:
        score += 2; rules.append("R2: electronics > 1500")

    # YOUR CODE — add rule R3: hour < 6 (use 'hour' field if present in tx)

    if score >= 5:
        risk_level = "CRITICAL"
    elif score >= 3:
        risk_level = "HIGH"
    elif score >= 1:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    return {"score": score, "risk_level": risk_level, "triggered_rules": rules}

# Quick test without a server
test_tx = {"tx_id": "TX001", "amount": 4500.0, "category": "electronics", "hour": 3}
print(score_transaction(test_tx))

In [ ]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

def score_transaction(tx):
    score = 0
    rules = []
    if tx.get("amount", 0) > 3000:
        score += 3; rules.append("R1: amount > 3000")
    if tx.get("category") == "electronics" and tx.get("amount", 0) > 1500:
        score += 2; rules.append("R2: electronics > 1500")
    if tx.get("hour", 12) < 6:
        score += 2; rules.append("R3: night hour")
    risk_level = "CRITICAL" if score >= 5 else "HIGH" if score >= 3 else "MEDIUM" if score >= 1 else "LOW"
    return {"score": score, "risk_level": risk_level, "triggered_rules": rules}

@app.route("/score", methods=["POST"])
def score():
    tx = request.get_json()
    if not tx or "amount" not in tx:
        return jsonify({"error": "Missing required field 'amount'"}), 400
    result = score_transaction(tx)
    result["tx_id"] = tx.get("tx_id", "unknown")
    return jsonify(result)

@app.route("/health")
def health():
    return jsonify({"status": "ok", "version": "1.0-rules"})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

In [ ]:
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

cases = [
    {"tx_id": "TX001", "amount": 50.0,   "category": "food",        "hour": 14},
    {"tx_id": "TX002", "amount": 1800.0,  "category": "electronics", "hour": 10},
    {"tx_id": "TX003", "amount": 4500.0,  "category": "electronics", "hour": 3},
]

for tx in cases:
    r = requests.post("http://localhost:5000/score", json=tx)
    res = r.json()
    print(f"{tx['tx_id']}  {tx['amount']:>7.0f} PLN  → {res['risk_level']:8s} (score={res['score']})  {res['triggered_rules']}")

In [ ]:
#4.3
r = requests.post(
    "http://localhost:5000/score",
    json={"tx_id": "TX000"}
)

print("Status:", r.status_code)
print("Response:", r.json())

In [ ]:
#4.4
#1. # ANSWER:
# GET is used to retrieve data from a server.
# POST is used to send data to a server for processing.
# GET parameters are usually included in the URL,
# while POST data is sent in the request body.

#2. Why use jsonify() instead of return {"key": "value"}?
# ANSWER:
# jsonify() converts Python objects into proper JSON responses and automatically sets the Content-Type header to application/json.
# It ensures clients correctly interpret the response as JSON.

#3 What happens if two people call /score at the same time?
# ANSWER:
# Flask will process both requests independently.
# Each request gets its own transaction data and response.
# In production, multiple worker processes or threads can handle
# many concurrent requests simultaneously.

In [ ]:
server.kill()
print("Server stopped.")

In [ ]:
### HOMEWORK 

In [ ]:
#1. Add endpoint GET /stats that returns: how many requests the server handled, how many were HIGH/CRITICAL.
#(Hint: use a global dict as a counter — counters = {"total": 0, "high": 0})

In [ ]:
%%file app.py
from flask import Flask, request, jsonify

app = Flask(__name__)

counters = {"total": 0, "high": 0}

def score_transaction(tx):
    score = 0
    rules = []

    if tx.get("amount", 0) > 3000:
        score += 3
        rules.append("R1: amount > 3000")

    if tx.get("category") == "electronics" and tx.get("amount", 0) > 1500:
        score += 2
        rules.append("R2: electronics > 1500")

    if tx.get("hour", 12) < 6:
        score += 2
        rules.append("R3: night hour")

    risk_level = (
        "CRITICAL" if score >= 5
        else "HIGH" if score >= 3
        else "MEDIUM" if score >= 1
        else "LOW"
    )

    return {
        "score": score,
        "risk_level": risk_level,
        "triggered_rules": rules
    }

@app.route("/score", methods=["POST"])
def score():
    tx = request.get_json()

    result = score_transaction(tx)

    counters["total"] += 1

    if result["risk_level"] in ["HIGH", "CRITICAL"]:
        counters["high"] += 1

    return jsonify(result)

@app.route("/stats")
def stats():
    return jsonify(counters)

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

In [ ]:
server.kill()
server = subprocess.Popen(["python", "app.py"])
time.sleep(2)

for tx in [
    {"amount": 50},
    {"amount": 4000},
    {"amount": 5000, "category": "electronics", "hour": 2}
]:
    requests.post("http://localhost:5000/score", json=tx)

r = requests.get("http://localhost:5000/stats")
print(r.json())

In [ ]:
##2.Test validation (negative amount)
r = requests.post(
    "http://localhost:5000/score",
    json={
        "tx_id": "BAD001",
        "amount": -100
    }
)

print(r.status_code)
print(r.json())

In [ ]:
#3. Call /score with 10 transactions
import requests
import random

results = []

for i in range(10):

    tx = {
        "tx_id": f"TX{i+1:03d}",
        "amount": random.randint(10, 5000),
        "category": random.choice(
            ["electronics", "food", "books"]
        ),
        "hour": random.randint(0, 23)
    }

    r = requests.post(
        "http://localhost:5000/score",
        json=tx
    )

    res = r.json()

    results.append([
        tx["tx_id"],
        tx["amount"],
        res["score"],
        res["risk_level"]
    ])

In [ ]:
print(
    f"{'TX_ID':<8} {'AMOUNT':<10} {'SCORE':<8} {'RISK'}"
)

for row in results:
    print(
        f"{row[0]:<8} {row[1]:<10} {row[2]:<8} {row[3]}"
    )